<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabianalexisbernabecruz-code/portafolio-actuaria-unam/blob/main/15_Numerical_Analysis_Project.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717?logo=github)](https://github.com/fabianalexisbernabecruz-code/portafolio-actuaria-unam)

</div>



<font face="Times New Roman">

 <img src="https://upload.wikimedia.org/wikipedia/commons/c/ca/Escudo-UNAM-escalable.svg" width="150" height="170" align="left"/>

  <img src="https://upload.wikimedia.org/wikipedia/commons/5/5b/Escudo-facultad-ciencias-unam-escalable.svg" width="150" height="170" align="right"/>

<h5 align="center"><i> </i></h5>
<h1 align="center"><b>Modelación Numérica de la Curva Cupón Cero</b></h1>
</font>
<font face="Times New Roman">

---
---

<h2 align="center"><i>Proyecto de Análisis Numérico</i></h2>
<h5 align="center"><i>Implementación de interpolación y diferenciación numérica aplicada al modelado de curvas financieras.</i></h5>

---
---

<h5 align="center"><i>Universidad Nacional Autónoma de México</i></h5>
<h5 align="center"><i>Facultad de Ciencias — Actuaría</i></h5>
<h5 align="center"><i>Materia: Análisis Numérico</i></h5>

---
---

<font color="Black" face="Times New Roman">
<h5 align="center"><i>Bernabe Cruz Fabian Alexis</i></h5>

</font>
</font>

<font face="Times New Roman">

## Problema: Construcción de una curva cupón cero.

<font face="Times New Roman">

Considera la siguiente curva que contiene bonos cupón cero (ZC) y bonos cuponados (BC), crea una curva cupón cero.


<div align="center">

| Plazo     | YTM | Tipo Cupón | Tasa Cupón |
|-----------|------|------------|------------|
| 6 meses   | 10.2 | ZC         | 0          |
| 1 año     | 10.3 | BC         | 8          |
| 2 años    | 10.35| BC         | 8          |
| 3.5 años  | 10.40| BC         | 8          |
| 5 años    | 10.50| BC         | 8          |


</div>

</font>

<font face="Times New Roman">

**Solución:**

</font>

<font face="Times New Roman">

Para utilizar el método de *bootstrapping* es necesario que los plazos de vencimiento de los cupones sean equidistantes, por ello podemos usar un spline cúbico para interpolar los bonos faltantes y obtener una curva que sí podamos usar para aplicar *bootstrapping*

</font>

<font face="Times New Roman">

**a) Interpolación**

</font>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

# Datos
x = np.array([0.5, 1.0, 2.0, 3.5, 5.0])
y = np.array([10.200000, 10.300000, 10.350000, 10.400000, 10.500000])

# Creamos el spline cúbico
cs = CubicSpline(x, y)

# Creamos puntos para graficar el spline
x_new = np.linspace(min(x), max(x), 500) # valores arbitrarios en X que tengan a los valores de x
y_new = cs(x_new)                        # los valores calculados por CubicSpline tomando los valores de x_new

# Graficamos los datos y el spline
plt.figure(figsize=(8, 6))
plt.plot(x, y, 'o', color='red', label="Datos originales")
plt.plot(x_new, y_new, '-', color='gray', label="Spline cúbico")
plt.xlabel("Plazo (Años)")
plt.ylabel("YTM")
plt.title("Interpolación con Splines Cúbicos")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Podemos obtener los cupones intermedios entre las fechas dadas
x_especificos = np.array([1.5, 2.5, 3, 4, 4.5])  # Puntos donde queremos evaluar el spline
y_especificos = cs(x_especificos)      # Valores interpolados correspondientes
# Imprimimos resultados
for xi, yi in zip(x_especificos, y_especificos):
    print(f"En x = {xi}, el valor interpolado del spline es y = {yi:.6f}")

<font face="Times New Roman">

---

</font>

<font face="Times New Roman">

**(b) Bootstraping**.

</font>

<font face="Times New Roman">

Obteniendo con lo anterior la siguiente tabla:

<div align="center">

| Plazo     | Tasa     | Tipo Cupón | Tasa cupón |
|-----------|----------|------------|------------|
| 6 meses   | 10.200000| ZC         | 0          |
| 1 año     | 10.300000| BC         | 8          |
| 1.5 años  | 10.338889| BC         | 8          |
| 2 años    | 10.350000| BC         | 8          |
| 2.5 años  | 10.361111| BC         | 8          |
| 3 años    | 10.377778| BC         | 8          |
| 3.5 años  | 10.400000| BC         | 8          |
| 4 años    | 10.427778| BC         | 8          |
| 4.5 años  | 10.461111| BC         | 8          |
| 5 años    | 10.500000| BC         | 8          |

</div>

</font>

<font face="Times New Roman">

Usando lo anterior podemos

</font>

<font face="Times New Roman">

Para calcular la tasa cupón cero:

$$
TZ_{n} = \left( \frac{VN + C}{P - \sum_{i=1}^{n-1} \frac{C}{(1 + TZ_{i})^{i}}} \right)^{\frac{1}{n}} - 1, \quad n=2,3,\ldots
$$

</font>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
# Creamos la curva
curva = {
    'Plazo': ['6 meses', '1 año', '1.5 años', '2 años', '2.5 años', '3 años', '3.5 años', '4 años', '4.5 años', '5 años'],
    'Tasa': [10.200000, 10.300000, 10.338889, 10.350000, 10.361111, 10.377778, 10.400000, 10.427778, 10.461111, 10.500000],
    'Tipo Cupón': ['ZC', 'BC', 'BC', 'BC', 'BC', 'BC', 'BC', 'BC', 'BC', 'BC'],
    'Tasa cupón': [0, 8, 8, 8, 8, 8, 8, 8, 8, 8],
    'Plazo_num': [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
}

df_curva = pd.DataFrame(curva)

In [ ]:
# Para el método necesitamos los precios de cada bono, por lo que creamos una función que calcule el precio sucio.
def calcular_precio_sucio(nominal, tasa_cupon, plazo_num, tasa_descuento, tipo_cotizacion):
    '''
    Calcula el precio sucio de un bono, ya sea un bono cupón cero (ZC) o un bono con cupones (BC).

    Parámetros:
    - nominal (float): Valor nominal del bono.
    - tasa_cupon (float): Tasa de cupón anualizada del bono.
    - plazo_num (float): Plazo de vencimiento del bono (en años).
    - tasa_descuento (float): Tasa de descuento (Yield to Maturity) anualizada, utilizada para descontar los pagos futuros.

    Retorna:
    - precio (float): El precio sucio del bono.

    '''
    precio = 0
    # Tenemos dos casos; uno para cada tipo de cotización
    if tipo_cotizacion == "30/360":
        factor_tiempo = 180 / 360
    elif tipo_cotizacion == "Actual/Actual":
        factor_tiempo = 182 / 365
    else:
        raise ValueError("Tipo de cotización no válido. Use '30/360' o 'Actual/Actual'.")

    # Cálculo para bonos cupón cero (ZC) consideramos que es una tasa spot
    if tasa_cupon == 0:
        precio = nominal / (1 + (1 + tasa_descuento * factor_tiempo))**plazo_num
    else:  # Cálculo para bonos cuponados (BC)
        # obtenemos el cupón del bono
        cupon_periodo = (tasa_cupon) * nominal
        # iteramos para descontar cada cupón
        for i in range(1, int(plazo_num * 2) + 1):
            precio += cupon_periodo / (1 + tasa_descuento * factor_tiempo) ** (i * factor_tiempo)  # Ajustado para cupones semestrales
        # descontamos el valor nominal
        precio += nominal / (1 + tasa_descuento * factor_tiempo) ** plazo_num
    # regresamos el precio sucio del bono
    return precio

# El precio que obtenemos con la función de antes y los datos de la curva son
# suficientes para calcular cada nueva tasa cupón cero (TZ)

def calcular_tz(nominal, cupon_bono, precio_bono, plazo_num, tasas_cero_previas):
        '''
    Calcula la tasa cupón cero (TZ) de un bono en un plazo determinado utilizando el método de bootstraping.

    Parámetros:
    - nominal (float): Valor nominal del bono.
    - cupon_bono (float): Monto del cupón del bono.
    - precio_bono (float): Precio del bono.
    - plazo_num (float): Plazo de vencimiento del bono en años.
    - tasas_cero_previas (list): Lista de tasas cupón cero (TZ) previamente calculadas para bonos anteriores, usadas para descontar
      los pagos de cupones intermedios.

    Retorno:
    - tz (float): La nueva tasa cupón cero (TZ) para el bono en el plazo especificado, en decimales y semestral.

    '''
        # necesitamos descontar los cupones del bono usando ahora las tasas spot
        suma_descuentos = sum(cupon_bono / (1 + tasas_cero_previas[i])**df_curva.loc[i, 'Plazo_num']
                            for i in range(len(tasas_cero_previas)))
        # usamos la formula mencionada al inicio
        tz = ((nominal + cupon_bono) / (precio_bono - suma_descuentos)) ** (1 / plazo_num) - 1
        return tz

# Función principal para generar la curva cupón cero
def generar_curva_cupon_cero(curva_df, tipo_cotizacion):
    """
    Calcula la curva cupón cero para una serie de bonos, ajustando los precios y tasas
    según el tipo de cotización.

    Parámetros:
    -curva_df : pandas.DataFrame
        DataFrame con la curva de bonos, que incluye las columnas 'Tasa', 'Tasa cupón',
        'Plazo_num' y 'Tipo Cupón' ('ZC' para cupón cero).
    -tipo_cotizacion : str
        Tipo de cotización: '30/360' (cada mes de 30 días y año de 360) o 'Actual/Actual'.
    Retorno:
    pandas.DataFrame
        `curva_df` modificado con las columnas adicionales 'Precio' y 'Tasa Cupón Cero' y gráfica de la curva cupón cero.
    """
    nominal = 100  # Valor nominal estándar para los bonos
    tasas_cero = []

    for i, row in curva_df.iterrows():
        tasa_descuento = row['Tasa'] / 100
        precio_bono = calcular_precio_sucio(nominal, row['Tasa cupón'] / 100, row['Plazo_num'], tasa_descuento, tipo_cotizacion)

        if row['Tipo Cupón'] == 'ZC':
            tz = tasa_descuento / 2  # Tasa cupón cero directa para bonos ZC
        else:
            tz = calcular_tz(nominal, row['Tasa cupón'] / 100 * nominal, precio_bono, row['Plazo_num'], tasas_cero)

        tasas_cero.append(tz)
        curva_df.loc[i, 'Precio'] = precio_bono
        curva_df.loc[i, 'Tasa Cupón Cero'] = tz

    # Mostramos la tabla
    display(curva_df)

    # Graficamos la curva cupón cero
    plt.figure(figsize=(10, 6))
    plt.plot(curva_df['Plazo_num'], [tz * 100 for tz in curva_df['Tasa Cupón Cero']], marker='o', color='#1f77b4', linewidth=2)
    plt.xlabel('Plazo (años)', fontsize=12, fontweight='bold')
    plt.ylabel('Tasa Cupón Cero (%)', fontsize=12, fontweight='bold')
    plt.title('Curva Cupón Cero', fontsize=14, fontweight='bold')
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(visible=True, linestyle='--', linewidth=0.5, alpha=0.7)
    plt.show()

    return curva_df

In [ ]:
# Aplicado a lo que queremos:
df_curva = pd.DataFrame(curva)
df_resultado = generar_curva_cupon_cero(df_curva, 'Actual/Actual')

<font face="Times New Roman">

# Conclusiones:

*Aplicación de Splines Cúbicos:*

Se utilizó la técnica de splines cúbicos para interpolar las tasas faltantes en la curva de rendimiento. Este método fue útil porque proporcionó una aproximación suave y continua que garantiza la continuidad hasta la segunda derivada. Es esencial en el ámbito financiero, donde los cambios en las tasas pueden hacer que los cálculos sean inexactos para decisiones de inversión.


*Ventajas de los Splines sobre Otros Métodos:*

Comparado con métodos de interpolación, los splines cúbicos son más estables, especialmente cuando se trabaja con datos distribuidos de manera no uniforme. Esto permite obtener una representación precisa de la curva de tasas con base en los puntos conocidos, como se observó al calcular los valores interpolados en los plazos intermedios.

*Precisión y Eficiencia Numérica:*


El uso de splines cúbicos hizo los cálculos más eficientes y evitó complicaciones que podrían surgir con métodos más complejos. Además, aseguró que las tasas calculadas fueran precisas y estuvieran en línea con los datos originales.

</font>